# Model Merging — Inference with Ground Truth
Uses `allenai/ai2_arc` (ARC-Challenge) for science and `cais/mmlu` (high_school_mathematics) for math.
Both datasets are MCQA with ground truth answer keys, so accuracy can be computed directly.

In [ ]:
!nvidia-smi

## Install Packages

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q datasets==3.5.1 bitsandbytes>=0.46.1 huggingface_hub>=1.3.0

## Import Packages

In [ ]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

import json, os, re
import numpy as np
import torch
from tqdm import tqdm
from datasets import load_dataset
from peft import PeftModel, LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig

## Load Datasets with Ground Truth
- **Math**: `cais/mmlu` → `high_school_mathematics` test split (100 questions, MCQA, answerKey 0-3)
- **Science**: `allenai/ai2_arc` → `ARC-Challenge` test split (MCQA, answerKey A-D)

In [ ]:
N_SAMPLES = 200  # set to None to use the full test split

# ── Math: MMLU high school mathematics ──
mmlu_raw = load_dataset("cais/mmlu", "high_school_mathematics", split="test")
if N_SAMPLES:
    mmlu_raw = mmlu_raw.select(range(min(N_SAMPLES, len(mmlu_raw))))

OPTION_LABELS = ["A", "B", "C", "D"]
math_data = []
for i, item in enumerate(mmlu_raw):
    math_data.append({
        "id": f"mmlu_math_{i}",
        "question": item["question"],
        "options": {OPTION_LABELS[j]: item["choices"][j] for j in range(4)},
        "answer": OPTION_LABELS[item["answer"]],  # ground truth
    })

# ── Science: ARC-Challenge ──
arc_raw = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
if N_SAMPLES:
    arc_raw = arc_raw.select(range(min(N_SAMPLES, len(arc_raw))))

science_data = []
for i, item in enumerate(arc_raw):
    labels = item["choices"]["label"]
    texts  = item["choices"]["text"]
    # Normalize labels: some ARC items use 1/2/3/4 instead of A/B/C/D
    norm = {"1":"A","2":"B","3":"C","4":"D"}
    options = {norm.get(l, l): t for l, t in zip(labels, texts)}
    ak = item["answerKey"]
    ak = norm.get(ak, ak)
    science_data.append({
        "id": f"arc_{i}",
        "question": item["question"],
        "options": options,
        "answer": ak,  # ground truth
    })

print(f"Math (MMLU)   : {len(math_data)} questions")
print(f"Science (ARC) : {len(science_data)} questions")
print(f"\nSample Math   : {math_data[0]}")
print(f"\nSample Science: {science_data[0]}")

## Save Datasets Locally

In [ ]:
with open("/content/MATH.json", "w") as f:
    json.dump(math_data, f, indent=2)
with open("/content/SCIENCE.json", "w") as f:
    json.dump(science_data, f, indent=2)
print("Saved MATH.json and SCIENCE.json")

## Prompt Generation

In [ ]:
INSTRUCTION = {
    "MATH":    "You are given a math question and four answer options (associated with \"A\", \"B\", \"C\", \"D\"). Your task is to carefully analyze the problem, apply logical reasoning, and select the correct answer. There is only one correct answer for each question.",
    "SCIENCE": "You are given a science question and four answer options (associated with \"A\", \"B\", \"C\", \"D\"). Your task is to find the correct answer based on scientific facts, knowledge, and reasoning. There is only one correct answer for each question.",
}

def build_prompt(task_name, data_point):
    sys_msg = INSTRUCTION[task_name]
    opts = data_point["options"]
    options_str = " ".join([f"({k}) {v}" for k, v in opts.items()])
    user_prompt = f"Question: {data_point['question']}; Options: {options_str}"
    return f"""[INST] <<SYS>>You are a helpful assistant and good at solving tasks based on task instructions and inputs.<</SYS>> Instruction: {sys_msg} Input: {user_prompt}[/INST]"""

## Seed & Device Settings

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda:0")
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
np.random.seed(42)

## Merging Settings

In [ ]:
#### TODO: adjust these ####
MERGE_TYPE   = "dare_ties"   # linear | magnitude_prune | ties | dare_ties | dare_linear
density      = 0.2
weight_math  = 1.0
weight_sci   = 0.4
weights      = [weight_math, weight_sci]

OUTPUT_DIR = "/content/drive/MyDrive/ml2025_hw9/outputs_gt"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load Model & Adapters

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
lora_config = LoraConfig(
    r=8, target_modules=["q_proj","k_proj","v_proj"],
    task_type="CAUSAL_LM", lora_alpha=16, lora_dropout=0.05
)

base_model_name = "unsloth/llama-2-7b-chat-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
tokenizer.pad_token_id = 1
tokenizer.eos_token_id = 2

model = AutoModelForCausalLM.from_pretrained(
    base_model_name, quantization_config=quant_config,
    low_cpu_mem_usage=True, torch_dtype=torch.float16
).to(device)
model.resize_token_embeddings(len(tokenizer))

hf_adapters = {
    "MATH":    "MonicaHuang/llama-2-7b-chat-GSM8K-MCQA",
    "SCIENCE": "chenjoachim/llama-2-7b-chat-ARC-MCQA",
}
TASK_NAMES = ["MATH", "SCIENCE"]

model = PeftModel.from_pretrained(model, hf_adapters[TASK_NAMES[0]], adapter_name=TASK_NAMES[0]).to(device)
for t in TASK_NAMES[1:]:
    model.load_adapter(hf_adapters[t], adapter_name=t, device=device)

print("Model and adapters loaded.")

## Apply Merging Algorithm

In [ ]:
if MERGE_TYPE == "ties":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="ties", density=density)
elif MERGE_TYPE == "linear":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="linear")
elif MERGE_TYPE == "magnitude_prune":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="magnitude_prune", density=density)
elif MERGE_TYPE == "dare_ties":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="dare_ties", density=density)
elif MERGE_TYPE == "dare_linear":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="dare_linear", density=density)

model.set_adapter("merge")
print(f"Merged with: {MERGE_TYPE} | density={density} | weights={weights}")

## Generation Config

In [ ]:
hyperparameters = {
    "do_sample": False, "temperature": None,
    "num_beams": 1, "top_p": None,
    "no_repeat_ngram_size": 3, "max_new_len": 400
}
generation_config = GenerationConfig(
    do_sample=hyperparameters["do_sample"],
    temperature=hyperparameters["temperature"],
    top_p=hyperparameters["top_p"],
    num_beams=hyperparameters["num_beams"],
    pad_token_id=1,
    max_new_tokens=hyperparameters["max_new_len"]
)
print("Generation config set.")

## Inference Helper

In [ ]:
def run_inference(task_name, test_datas):
    results = []
    for data_point in tqdm(test_datas, desc=task_name):
        prompt = build_prompt(task_name, data_point)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
        input_ids = inputs["input_ids"].cuda()

        model.eval()
        with torch.no_grad():
            output = model.generate(
                input_ids=input_ids,
                generation_config=generation_config,
                return_dict_in_generate=True,
                output_scores=True,
                max_new_tokens=hyperparameters["max_new_len"],
            )
        for s in output.sequences:
            predict = tokenizer.decode(s)
        response = predict.split("[/INST]")[-1].split("</s>")[0].strip()

        results.append({
            "id":       data_point["id"],
            "response": response,
            "answer":   data_point["answer"],   # ground truth stored alongside
        })
    return results

## Run Math Inference (MMLU)

In [ ]:
wnames = '_'.join([str(float(w)) for w in weights])

math_results = run_inference("MATH", math_data)

result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}/MATH"
os.makedirs(result_dir, exist_ok=True)
out_path = f"{result_dir}/w_{wnames}_d_{density}_result.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(math_results, f, indent=2, ensure_ascii=False)
print(f"Saved → {out_path}")

## Run Science Inference (ARC-Challenge)

In [ ]:
science_results = run_inference("SCIENCE", science_data)

result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}/SCIENCE"
os.makedirs(result_dir, exist_ok=True)
out_path = f"{result_dir}/w_{wnames}_d_{density}_result.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(science_results, f, indent=2, ensure_ascii=False)
print(f"Saved → {out_path}")

## Compute Accuracy

In [ ]:
def extract_answer(response):
    matches = re.findall(r'\(([A-D])\)', response)
    return matches[-1] if matches else None

def accuracy(results):
    correct = no_pred = 0
    for r in results:
        pred = extract_answer(r["response"])
        if pred is None:
            no_pred += 1
        correct += (pred == r["answer"])
    total = len(results)
    return correct / total, correct, total, no_pred

math_acc, mc, mt, mn   = accuracy(math_results)
sci_acc,  sc, st, sn   = accuracy(science_results)
avg = (math_acc + sci_acc) / 2

print(f"Merge  : {MERGE_TYPE}  |  weights={weights}  |  density={density}")
print(f"Math   (MMLU)         : {mc}/{mt} = {math_acc:.2%}  (no-pred: {mn})")
print(f"Science (ARC-Challenge): {sc}/{st} = {sci_acc:.2%}  (no-pred: {sn})")
print(f"Average               : {avg:.2%}")